# 입문자를 위한, 파이썬/R 데이터 분석      

]

###  (공공데이터분석) 국토부 아파트매매 실거래가 API[data.go.kr] 연동

- XML 국토교통부_아파트매매 실거래자료 : https://www.data.go.kr/data/15126469/openapi.do
- 행정표준관리시스템 : https://www.code.go.kr/index.do ( [메뉴] 자주이용하는 코드 > 법정동 )
- 개별적 실습을 위해 회원가입 및 API 발급 필요

]

### 1. Import Packages

In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup

### 2. Data Loading 

#### : API Key Setting

In [2]:
#f=open("C:/Users/USER/Desktop/연습데이터/WEEK03_01_공공데이터수집/data/api_key.txt","r")
#lines=f.readlines()
#my_key=lines[0]
#f.close()

my_key = pd.read_csv("C:/Users/USER/Desktop/연습데이터/WEEK03_01_공공데이터수집/data/api_key.txt",
                     header=None).iloc[0,0]

In [3]:
my_key

'accc226aba90e01bb632eb0bc06e1fe71b711c4f1144a8512c3a589482c0c514'

#### : Pre-defined

In [4]:
#"http://openapi.molit.go.kr:8081/OpenAPI_ToolInstallPackage/service/rest/RTMSOBJSvc/getRTMSDataSvcAptTrade
#        ?
#        LAWD_CD=11110
#        &
#        DEAL_YMD=201512
#        &
#        serviceKey=서비스키"

In [5]:
base_url = "https://apis.data.go.kr/1613000/RTMSDataSvcAptTrade/getRTMSDataSvcAptTrade"

In [6]:
area_cd = "LAWD_CD=" + "11680"    # 11680 : 강남구
deal_dt = "DEAL_YMD=" + "202405"
svc_key = "serviceKey=" + my_key

In [7]:
url = base_url + "?" + area_cd + "&" + deal_dt + "&" + svc_key
url

'https://apis.data.go.kr/1613000/RTMSDataSvcAptTrade/getRTMSDataSvcAptTrade?LAWD_CD=11680&DEAL_YMD=202405&serviceKey=accc226aba90e01bb632eb0bc06e1fe71b711c4f1144a8512c3a589482c0c514'

#### : Request Test

In [8]:
response = requests.get(url)
response

<Response [200]>

In [9]:
response.text

'<?xml version="1.0" encoding="utf-8" standalone="yes"?><response><header><resultCode>000</resultCode><resultMsg>OK</resultMsg></header><body><items><item><aptDong>109</aptDong><aptNm>신현대11차</aptNm><buildYear>1983</buildYear><buyerGbn>개인</buyerGbn><cdealDay> </cdealDay><cdealType> </cdealType><dealAmount>690,000</dealAmount><dealDay>17</dealDay><dealMonth>5</dealMonth><dealYear>2024</dealYear><dealingGbn>중개거래</dealingGbn><estateAgentSggNm>서울 강남구</estateAgentSggNm><excluUseAr>183.41</excluUseAr><floor>3</floor><jibun>433</jibun><landLeaseholdGbn>N</landLeaseholdGbn><rgstDate>24.10.31</rgstDate><sggCd>11680</sggCd><slerGbn>법인</slerGbn><umdNm>압구정동</umdNm></item><item><aptDong>101</aptDong><aptNm>아크로힐스논현</aptNm><buildYear>2014</buildYear><buyerGbn>개인</buyerGbn><cdealDay> </cdealDay><cdealType> </cdealType><dealAmount>201,000</dealAmount><dealDay>31</dealDay><dealMonth>5</dealMonth><dealYear>2024</dealYear><dealingGbn>중개거래</dealingGbn><estateAgentSggNm>서울 강남구</estateAgentSggNm><excluUseAr>8

### Addtional Preparation : 

#### -1)  Input Data : lawd_cd / 법정동 코드(지역코드) 가공

In [10]:
lawd_cd = pd.read_csv("C:/Users/USER/Desktop/연습데이터/WEEK03_01_공공데이터수집/data/법정동코드 전체자료.txt",
                      sep='\t', encoding ='cp949') #'euc-kr', 'utf-8'
lawd_cd

,법정동코드,법정동명,폐지여부
0,1100000000,서울특별시,존재
1,1111000000,서울특별시 종로구,존재
2,1111010100,서울특별시 종로구 청운동,존재
3,1111010200,서울특별시 종로구 신교동,존재
4,1111010300,서울특별시 종로구 궁정동,존재
...,...,...,...
49856,5280042024,전북특별자치도 부안군 위도면 대리,존재
49857,5280042025,전북특별자치도 부안군 위도면 거륜리,존재
49858,5280042026,전북특별자치도 부안군 위도면 식도리,존재
49859,5280042027,전북특별자치도 부안군 위도면 상왕등리,존재


In [11]:
# Column 이름 변경 : 데이터 조작 편의성을 위함
lawd_cd.columns

Index(['법정동코드', '법정동명', '폐지여부'], dtype='object')

In [12]:
lawd_cd.columns = ["code", "name", "check"]
lawd_cd

,code,name,check
0,1100000000,서울특별시,존재
1,1111000000,서울특별시 종로구,존재
2,1111010100,서울특별시 종로구 청운동,존재
3,1111010200,서울특별시 종로구 신교동,존재
4,1111010300,서울특별시 종로구 궁정동,존재
...,...,...,...
49856,5280042024,전북특별자치도 부안군 위도면 대리,존재
49857,5280042025,전북특별자치도 부안군 위도면 거륜리,존재
49858,5280042026,전북특별자치도 부안군 위도면 식도리,존재
49859,5280042027,전북특별자치도 부안군 위도면 상왕등리,존재


In [13]:
lawd_cd.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49861 entries, 0 to 49860
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   code    49861 non-null  int64 
 1   name    49861 non-null  object
 2   check   49861 non-null  object
dtypes: int64(1), object(2)
memory usage: 1.1+ MB


In [14]:
in_name = "남구"

In [15]:
lawd_cd['name'].str.contains(in_name)

0        False
1        False
2        False
3        False
4        False
         ...  
49856    False
49857    False
49858    False
49859    False
49860    False
Name: name, Length: 49861, dtype: bool

In [16]:
lawd_cd['check'] == "존재"

0        True
1        True
2        True
3        True
4        True
         ... 
49856    True
49857    True
49858    True
49859    True
49860    True
Name: check, Length: 49861, dtype: bool

In [17]:
(lawd_cd['name'].str.contains(in_name)) & (lawd_cd['check'] == "존재")

0        False
1        False
2        False
3        False
4        False
         ...  
49856    False
49857    False
49858    False
49859    False
49860    False
Length: 49861, dtype: bool

In [18]:
lawd_cd[(lawd_cd['name'].str.contains(in_name)) & (lawd_cd['check'] == "존재")]

,code,name,check
1003,1168000000,서울특별시 강남구,존재
1037,1168010100,서울특별시 강남구 역삼동,존재
1039,1168010300,서울특별시 강남구 개포동,존재
1040,1168010400,서울특별시 강남구 청담동,존재
1041,1168010500,서울특별시 강남구 삼성동,존재
...,...,...,...
31310,4711135000,경상북도 포항시 남구 호미곶면,존재
31311,4711135021,경상북도 포항시 남구 호미곶면 강사리,존재
31312,4711135022,경상북도 포항시 남구 호미곶면 대보리,존재
31313,4711135023,경상북도 포항시 남구 호미곶면 구만리,존재


In [19]:
tmp = lawd_cd[(lawd_cd['name'].str.contains('서울특별시')) &
              (lawd_cd['name'].str.contains(in_name)) & 
               (lawd_cd['check'] == "존재")].head(1)
tmp

,code,name,check
1003,1168000000,서울특별시 강남구,존재


In [20]:
tmp.loc[1003,'code']    # 가져오기는 하나 문제 : 인덱스 값을 알아야 함

1168000000

In [21]:
tmp.iloc[0,0]           # 인덱스로 데이터를 가져오도록 === 문제없음

1168000000

In [22]:
tmp = tmp.iloc[0,0]
print(tmp)
type(tmp)

1168000000


numpy.int64

In [23]:
tmp

1168000000

In [24]:
str(tmp)

'1168000000'

In [25]:
tmp.astype(str)

'1168000000'

In [26]:
tmp = tmp.astype(str)
tmp

'1168000000'

In [27]:
tmp[:5]

'11680'

In [28]:
lawd_cd[(lawd_cd['name'].str.contains(in_name)) & (lawd_cd['check'] == "존재")].head(1).iloc[0,0].astype(str)[0:5]

'11680'

In [29]:
# Function 작성 :
def find_lawd_cd(in_text):
    result = lawd_cd[(lawd_cd['name'].str.contains(in_text)) & (lawd_cd['check'] == "존재")].head(1).iloc[0,0].astype(str)[0:5]    
    return result

In [30]:
in_name = "강남구"
find_lawd_cd(in_name)

'11680'

In [31]:
find_lawd_cd('중구')

'11140'

#### -2)  Input Data : lawd_cd, deal_ymd 입력받기

In [37]:
in_lawd = input("검색지역 입력 : ")

검색지역 입력 : 강남구


In [38]:
in_lawd

'강남구'

In [39]:
in_deal_ymd = input("계약월 입력(예: 201912) : ")

계약월 입력(예: 201912) : 202506


In [40]:
in_deal_ymd

'202506'

In [41]:
in_svckey = my_key

In [42]:
in_lawd_cd = find_lawd_cd(in_lawd)
in_lawd_cd

'11680'

In [43]:
area_cd = "LAWD_CD=" + in_lawd_cd
deal_dt = "DEAL_YMD=" + in_deal_ymd
svc_key = "serviceKey=" + in_svckey

In [44]:
url = base_url + "?" + area_cd + "&" + deal_dt + "&" + svc_key
url

'https://apis.data.go.kr/1613000/RTMSDataSvcAptTrade/getRTMSDataSvcAptTrade?LAWD_CD=11680&DEAL_YMD=202506&serviceKey=accc226aba90e01bb632eb0bc06e1fe71b711c4f1144a8512c3a589482c0c514'

In [45]:
response = requests.get(url)
response

<Response [200]>

In [46]:
response.text

'<?xml version="1.0" encoding="utf-8" standalone="yes"?><response><header><resultCode>000</resultCode><resultMsg>OK</resultMsg></header><body><items><item><aptDong> </aptDong><aptNm>개포우성1</aptNm><buildYear>1983</buildYear><buyerGbn>개인</buyerGbn><cdealDay> </cdealDay><cdealType> </cdealType><dealAmount>620,000</dealAmount><dealDay>26</dealDay><dealMonth>6</dealMonth><dealYear>2025</dealYear><dealingGbn>중개거래</dealingGbn><estateAgentSggNm>서울 강남구</estateAgentSggNm><excluUseAr>158.54</excluUseAr><floor>5</floor><jibun>503</jibun><landLeaseholdGbn>N</landLeaseholdGbn><rgstDate> </rgstDate><sggCd>11680</sggCd><slerGbn>개인</slerGbn><umdNm>대치동</umdNm></item><item><aptDong> </aptDong><aptNm>한양2</aptNm><buildYear>1978</buildYear><buyerGbn>개인</buyerGbn><cdealDay> </cdealDay><cdealType> </cdealType><dealAmount>786,000</dealAmount><dealDay>30</dealDay><dealMonth>6</dealMonth><dealYear>2025</dealYear><dealingGbn>중개거래</dealingGbn><estateAgentSggNm>서울 강남구</estateAgentSggNm><excluUseAr>147.41</excluUseAr

### 3. Data Preprocessng 

#### : web data / API output

In [47]:
#### BS 시도 : 잘안됨
soup = BeautifulSoup(response.text, "lxml")
soup#print(soup.prettify())

C:\Users\USER\anaconda3\lib\site-packages\bs4\builder\__init__.py:545: XMLParsedAsHTMLWarning: It looks like you're parsing an XML document using an HTML parser. If this really is an HTML document (maybe it's XHTML?), you can ignore or filter this warning. If it's XML, you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the lxml package installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.
  warnings.warn(


<?xml version="1.0" encoding="utf-8" standalone="yes"?><html><body><response><header><resultcode>000</resultcode><resultmsg>OK</resultmsg></header><items><item><aptdong> </aptdong><aptnm>개포우성1</aptnm><buildyear>1983</buildyear><buyergbn>개인</buyergbn><cdealday> </cdealday><cdealtype> </cdealtype><dealamount>620,000</dealamount><dealday>26</dealday><dealmonth>6</dealmonth><dealyear>2025</dealyear><dealinggbn>중개거래</dealinggbn><estateagentsggnm>서울 강남구</estateagentsggnm><excluusear>158.54</excluusear><floor>5</floor><jibun>503</jibun><landleaseholdgbn>N</landleaseholdgbn><rgstdate> </rgstdate><sggcd>11680</sggcd><slergbn>개인</slergbn><umdnm>대치동</umdnm></item><item><aptdong> </aptdong><aptnm>한양2</aptnm><buildyear>1978</buildyear><buyergbn>개인</buyergbn><cdealday> </cdealday><cdealtype> </cdealtype><dealamount>786,000</dealamount><dealday>30</dealday><dealmonth>6</dealmonth><dealyear>2025</dealyear><dealinggbn>중개거래</dealinggbn><estateagentsggnm>서울 강남구</estateagentsggnm><excluusear>147.41</exclu

In [48]:
soup.find_all('item')

[<item><aptdong> </aptdong><aptnm>개포우성1</aptnm><buildyear>1983</buildyear><buyergbn>개인</buyergbn><cdealday> </cdealday><cdealtype> </cdealtype><dealamount>620,000</dealamount><dealday>26</dealday><dealmonth>6</dealmonth><dealyear>2025</dealyear><dealinggbn>중개거래</dealinggbn><estateagentsggnm>서울 강남구</estateagentsggnm><excluusear>158.54</excluusear><floor>5</floor><jibun>503</jibun><landleaseholdgbn>N</landleaseholdgbn><rgstdate> </rgstdate><sggcd>11680</sggcd><slergbn>개인</slergbn><umdnm>대치동</umdnm></item>,
 <item><aptdong> </aptdong><aptnm>한양2</aptnm><buildyear>1978</buildyear><buyergbn>개인</buyergbn><cdealday> </cdealday><cdealtype> </cdealtype><dealamount>786,000</dealamount><dealday>30</dealday><dealmonth>6</dealmonth><dealyear>2025</dealyear><dealinggbn>중개거래</dealinggbn><estateagentsggnm>서울 강남구</estateagentsggnm><excluusear>147.41</excluusear><floor>2</floor><jibun>493</jibun><landleaseholdgbn>N</landleaseholdgbn><rgstdate> </rgstdate><sggcd>11680</sggcd><slergbn>개인</slergbn><umdnm>압구

In [49]:
item = soup.find('item')
print(item)

<item><aptdong> </aptdong><aptnm>개포우성1</aptnm><buildyear>1983</buildyear><buyergbn>개인</buyergbn><cdealday> </cdealday><cdealtype> </cdealtype><dealamount>620,000</dealamount><dealday>26</dealday><dealmonth>6</dealmonth><dealyear>2025</dealyear><dealinggbn>중개거래</dealinggbn><estateagentsggnm>서울 강남구</estateagentsggnm><excluusear>158.54</excluusear><floor>5</floor><jibun>503</jibun><landleaseholdgbn>N</landleaseholdgbn><rgstdate> </rgstdate><sggcd>11680</sggcd><slergbn>개인</slergbn><umdnm>대치동</umdnm></item>


In [50]:
#item.find('거래금액')#.text

In [51]:
#### XML 파서로 시도
import xml.etree.ElementTree as ET

In [52]:
response#.content#.text

<Response [200]>

In [53]:
root = ET.fromstring(response.content)
root

<Element 'response' at 0x00000147500CE0E0>

In [54]:
item_list = []

In [55]:
root.find('body').find('items')[0]

<Element 'item' at 0x00000147500CE2C0>

In [56]:
type(root.find('body').find('items'))

xml.etree.ElementTree.Element

In [57]:
child = root.find('body').find('items')[0]
child

<Element 'item' at 0x00000147500CE2C0>

In [58]:
child.findall('*')#[0]

[<Element 'aptDong' at 0x00000147500CE310>,
 <Element 'aptNm' at 0x00000147500CE360>,
 <Element 'buildYear' at 0x00000147500CE3B0>,
 <Element 'buyerGbn' at 0x00000147500CE400>,
 <Element 'cdealDay' at 0x00000147500CE450>,
 <Element 'cdealType' at 0x00000147500CE4A0>,
 <Element 'dealAmount' at 0x00000147500CE4F0>,
 <Element 'dealDay' at 0x00000147500CE540>,
 <Element 'dealMonth' at 0x00000147500CE590>,
 <Element 'dealYear' at 0x00000147500CE5E0>,
 <Element 'dealingGbn' at 0x00000147500CE630>,
 <Element 'estateAgentSggNm' at 0x00000147500CE6D0>,
 <Element 'excluUseAr' at 0x00000147500CE720>,
 <Element 'floor' at 0x00000147500CE770>,
 <Element 'jibun' at 0x00000147500CE7C0>,
 <Element 'landLeaseholdGbn' at 0x00000147500CE860>,
 <Element 'rgstDate' at 0x00000147500CE8B0>,
 <Element 'sggCd' at 0x00000147500CE900>,
 <Element 'slerGbn' at 0x00000147500CE950>,
 <Element 'umdNm' at 0x00000147500CE9A0>]

In [59]:
element = child.findall('*')[0]

In [60]:
element.tag.strip()

'aptDong'

In [61]:
element.text.strip()

''

In [62]:
data = {}

In [63]:
tag = element.tag.strip()
text = element.text.strip()
data[tag] = text

In [64]:
data

{'aptDong': ''}

In [65]:
def find_xml_items(response):
    root = ET.fromstring(response.content)
    item_list = []
    for child in root.find('body').find('items'):
        elements = child.findall('*')
        data = {}
        for element in elements:
            tag = element.tag.strip()
            text = element.text.strip()
            data[tag] = text
        item_list.append(data)  
    return item_list

In [66]:
items_list = find_xml_items(response)
items_list

[{'aptDong': '',
  'aptNm': '개포우성1',
  'buildYear': '1983',
  'buyerGbn': '개인',
  'cdealDay': '',
  'cdealType': '',
  'dealAmount': '620,000',
  'dealDay': '26',
  'dealMonth': '6',
  'dealYear': '2025',
  'dealingGbn': '중개거래',
  'estateAgentSggNm': '서울 강남구',
  'excluUseAr': '158.54',
  'floor': '5',
  'jibun': '503',
  'landLeaseholdGbn': 'N',
  'rgstDate': '',
  'sggCd': '11680',
  'slerGbn': '개인',
  'umdNm': '대치동'},
 {'aptDong': '',
  'aptNm': '한양2',
  'buildYear': '1978',
  'buyerGbn': '개인',
  'cdealDay': '',
  'cdealType': '',
  'dealAmount': '786,000',
  'dealDay': '30',
  'dealMonth': '6',
  'dealYear': '2025',
  'dealingGbn': '중개거래',
  'estateAgentSggNm': '서울 강남구',
  'excluUseAr': '147.41',
  'floor': '2',
  'jibun': '493',
  'landLeaseholdGbn': 'N',
  'rgstDate': '',
  'sggCd': '11680',
  'slerGbn': '개인',
  'umdNm': '압구정동'},
 {'aptDong': '',
  'aptNm': '현대2',
  'buildYear': '1986',
  'buyerGbn': '개인',
  'cdealDay': '',
  'cdealType': '',
  'dealAmount': '413,000',
  'dealDay'

### 4. Result : Display & Save as file 

In [67]:
result = pd.DataFrame(items_list)

In [68]:
result

,aptDong,aptNm,buildYear,buyerGbn,cdealDay,cdealType,dealAmount,dealDay,dealMonth,dealYear,dealingGbn,estateAgentSggNm,excluUseAr,floor,jibun,landLeaseholdGbn,rgstDate,sggCd,slerGbn,umdNm
0,,개포우성1,1983,개인,,,"620,000",26,6,2025,중개거래,서울 강남구,158.54,5,503,N,,11680,개인,대치동
1,,한양2,1978,개인,,,"786,000",30,6,2025,중개거래,서울 강남구,147.41,2,493,N,,11680,개인,압구정동
2,,현대2,1986,개인,,,"413,000",27,6,2025,중개거래,서울 강남구,165.08,6,654,N,,11680,개인,개포동
3,15,은마,1979,개인,,,"378,000",17,6,2025,중개거래,서울 강남구,84.43,14,316,N,25.11.03,11680,개인,대치동
4,116,개포래미안포레스트,2020,개인,,,"289,000",22,6,2025,중개거래,서울 강남구,74.66,19,1282,N,25.10.23,11680,개인,개포동
5,1단지 113,삼성동힐스테이트 1단지,2008,개인,,,"348,000",24,6,2025,중개거래,서울 강남구,84.236,11,16-2,N,25.10.10,11680,개인,삼성동
6,10,"현대2차(10,11,20,23,24,25동)",1976,개인,,,"1,200,000",16,6,2025,중개거래,서울 강남구,196.84,8,369-1,N,25.10.02,11680,개인,압구정동
7,28,미성2차,1987,개인,,,"470,000",25,6,2025,중개거래,서울 강남구,74.4,3,397,N,25.10.24,11680,개인,압구정동
8,,선경2차(8동-12동),1985,개인,,,"530,000",26,6,2025,중개거래,서울 강남구,127.75,4,506,N,,11680,개인,대치동
9,104,디에이치포레센트,2021,개인,,,"290,000",27,6,2025,중개거래,서울 강남구,84.94,4,742,N,25.09.19,11680,개인,일원동


In [69]:
result.shape

(10, 20)

In [70]:
result.head()

,aptDong,aptNm,buildYear,buyerGbn,cdealDay,cdealType,dealAmount,dealDay,dealMonth,dealYear,dealingGbn,estateAgentSggNm,excluUseAr,floor,jibun,landLeaseholdGbn,rgstDate,sggCd,slerGbn,umdNm
0,,개포우성1,1983,개인,,,"620,000",26,6,2025,중개거래,서울 강남구,158.54,5,503,N,,11680,개인,대치동
1,,한양2,1978,개인,,,"786,000",30,6,2025,중개거래,서울 강남구,147.41,2,493,N,,11680,개인,압구정동
2,,현대2,1986,개인,,,"413,000",27,6,2025,중개거래,서울 강남구,165.08,6,654,N,,11680,개인,개포동
3,15,은마,1979,개인,,,"378,000",17,6,2025,중개거래,서울 강남구,84.43,14,316,N,25.11.03,11680,개인,대치동
4,116,개포래미안포레스트,2020,개인,,,"289,000",22,6,2025,중개거래,서울 강남구,74.66,19,1282,N,25.10.23,11680,개인,개포동


#### : file name

In [71]:
name_info = in_lawd + "_" + in_deal_ymd
name_info

'강남구_202506'

In [72]:
file_name = f"C:/Users/USER/Desktop/연습데이터/WEEK03_01_공공데이터수집/data/result_{name_info}.csv"
file_name

'C:/Users/USER/Desktop/연습데이터/WEEK03_01_공공데이터수집/data/result_강남구_202506.csv'

In [73]:
result.to_csv(file_name, index=False, encoding='cp949')

## Appendix : Function 

In [74]:
find_lawd_cd(in_lawd)

'11680'

In [75]:
find_xml_items(response)

[{'aptDong': '',
  'aptNm': '개포우성1',
  'buildYear': '1983',
  'buyerGbn': '개인',
  'cdealDay': '',
  'cdealType': '',
  'dealAmount': '620,000',
  'dealDay': '26',
  'dealMonth': '6',
  'dealYear': '2025',
  'dealingGbn': '중개거래',
  'estateAgentSggNm': '서울 강남구',
  'excluUseAr': '158.54',
  'floor': '5',
  'jibun': '503',
  'landLeaseholdGbn': 'N',
  'rgstDate': '',
  'sggCd': '11680',
  'slerGbn': '개인',
  'umdNm': '대치동'},
 {'aptDong': '',
  'aptNm': '한양2',
  'buildYear': '1978',
  'buyerGbn': '개인',
  'cdealDay': '',
  'cdealType': '',
  'dealAmount': '786,000',
  'dealDay': '30',
  'dealMonth': '6',
  'dealYear': '2025',
  'dealingGbn': '중개거래',
  'estateAgentSggNm': '서울 강남구',
  'excluUseAr': '147.41',
  'floor': '2',
  'jibun': '493',
  'landLeaseholdGbn': 'N',
  'rgstDate': '',
  'sggCd': '11680',
  'slerGbn': '개인',
  'umdNm': '압구정동'},
 {'aptDong': '',
  'aptNm': '현대2',
  'buildYear': '1986',
  'buyerGbn': '개인',
  'cdealDay': '',
  'cdealType': '',
  'dealAmount': '413,000',
  'dealDay'

In [76]:
def deal_by_lawd(cd_name, ymd):
    
    func_cd = find_lawd_cd(cd_name)
    
    svckey = my_key
    
    area_cd = "LAWD_CD=" + func_cd
    deal_dt = "DEAL_YMD=" + ymd
    svc_key = "serviceKey=" + svckey
    url = base_url + "?" + area_cd + "&" + deal_dt + "&" + svc_key

    response = requests.get(url)
    items_list = find_xml_items(response)
    result = pd.DataFrame(items_list)
    
    name_info = cd_name + "_" + ymd
    file_name = f"C:/Users/USER/Desktop/연습데이터/WEEK03_01_공공데이터수집/data/result_{name_info}.csv"

    result.to_csv(file_name, index=False, encoding='cp949')
    
    return file_name

In [77]:
#### - Input Data : LAWD_CD, DEAL_YMD, serviceKey  입력받기
func_gu = input("검색지역 입력 : ")
print(func_gu)

func_ymd = input("계약월 입력(예: 201911) : ")
print(func_ymd)

검색지역 입력 : 서울특별시
서울특별시
계약월 입력(예: 201911) : 202506
202506


In [78]:
deal_by_lawd(func_gu, func_ymd)

'C:/Users/USER/Desktop/연습데이터/WEEK03_01_공공데이터수집/data/result_서울특별시_202506.csv'

### 추가 실습 :
- 공공API 데이터 끌어오기 친숙해지기
- **(필수) 특정월에 서울시 전체 구의 데이터를 한번에 가져오기(입력값: 연월 202402)**
- (선택) 특정 서울시 하나의 구의 과거 1년치 데이터를 한번에 가져오기(입력값: 특정구 하나) 

- 기상청_중기예보 조회서비스 (추가 API 사용신청 필요)    
https://www.data.go.kr/tcs/dss/selectApiDataDetailView.do?publicDataPk=15059468

### 코드리뷰 과제(1)

- 서울시 모든 구의 리스트 추려내는 방법

In [79]:
lawd_cd

,code,name,check
0,1100000000,서울특별시,존재
1,1111000000,서울특별시 종로구,존재
2,1111010100,서울특별시 종로구 청운동,존재
3,1111010200,서울특별시 종로구 신교동,존재
4,1111010300,서울특별시 종로구 궁정동,존재
...,...,...,...
49856,5280042024,전북특별자치도 부안군 위도면 대리,존재
49857,5280042025,전북특별자치도 부안군 위도면 거륜리,존재
49858,5280042026,전북특별자치도 부안군 위도면 식도리,존재
49859,5280042027,전북특별자치도 부안군 위도면 상왕등리,존재


- 파이썬으로 구하는 방법

In [174]:
#1번 for 문
seoul = []
for i in range(len(lawd_cd)):
    if '서울특별시' in lawd_cd.loc[i, 'name']:
        seoul.append(i)
        
seoul_df = lawd_cd.loc[seoul]
    

In [175]:
seoul_df

,code,name,check
0,1100000000,서울특별시,존재
1,1111000000,서울특별시 종로구,존재
2,1111010100,서울특별시 종로구 청운동,존재
3,1111010200,서울특별시 종로구 신교동,존재
4,1111010300,서울특별시 종로구 궁정동,존재
...,...,...,...
1107,1174010600,서울특별시 강동구 둔촌동,존재
1108,1174010700,서울특별시 강동구 암사동,존재
1109,1174010800,서울특별시 강동구 성내동,존재
1110,1174010900,서울특별시 강동구 천호동,존재


In [179]:
#2번 for 문
mask = []
for i in lawd_cd['name']:
    mask.append('서울특별시' in i)
    
seoul_df = lawd_cd[mask]

In [180]:
seoul_df

,code,name,check
0,1100000000,서울특별시,존재
1,1111000000,서울특별시 종로구,존재
2,1111010100,서울특별시 종로구 청운동,존재
3,1111010200,서울특별시 종로구 신교동,존재
4,1111010300,서울특별시 종로구 궁정동,존재
...,...,...,...
1107,1174010600,서울특별시 강동구 둔촌동,존재
1108,1174010700,서울특별시 강동구 암사동,존재
1109,1174010800,서울특별시 강동구 성내동,존재
1110,1174010900,서울특별시 강동구 천호동,존재


- 판다스로 구하는 방법

In [ ]:
lawd_cd_seoule = lawd_cd[lawd_cd['name'].str.contains('서울특별시')]

In [ ]:
lawd_cd_seoule['name'].value_counts()

In [185]:
name_list = list(lawd_cd_seoule['name'])
name_list

['서울특별시',
 '서울특별시 종로구',
 '서울특별시 종로구 청운동',
 '서울특별시 종로구 신교동',
 '서울특별시 종로구 궁정동',
 '서울특별시 종로구 효자동',
 '서울특별시 종로구 창성동',
 '서울특별시 종로구 통의동',
 '서울특별시 종로구 적선동',
 '서울특별시 종로구 통인동',
 '서울특별시 종로구 누상동',
 '서울특별시 종로구 누하동',
 '서울특별시 종로구 옥인동',
 '서울특별시 종로구 체부동',
 '서울특별시 종로구 필운동',
 '서울특별시 종로구 내자동',
 '서울특별시 종로구 사직동',
 '서울특별시 종로구 도렴동',
 '서울특별시 종로구 당주동',
 '서울특별시 종로구 내수동',
 '서울특별시 종로구 세종로',
 '서울특별시 종로구 신문로1가',
 '서울특별시 종로구 신문로2가',
 '서울특별시 종로구 청진동',
 '서울특별시 종로구 서린동',
 '서울특별시 종로구 수송동',
 '서울특별시 종로구 중학동',
 '서울특별시 종로구 종로1가',
 '서울특별시 종로구 공평동',
 '서울특별시 종로구 관훈동',
 '서울특별시 종로구 견지동',
 '서울특별시 종로구 와룡동',
 '서울특별시 종로구 권농동',
 '서울특별시 종로구 운니동',
 '서울특별시 종로구 익선동',
 '서울특별시 종로구 경운동',
 '서울특별시 종로구 관철동',
 '서울특별시 종로구 인사동',
 '서울특별시 종로구 낙원동',
 '서울특별시 종로구 종로2가',
 '서울특별시 종로구 팔판동',
 '서울특별시 종로구 삼청동',
 '서울특별시 종로구 안국동',
 '서울특별시 종로구 소격동',
 '서울특별시 종로구 화동',
 '서울특별시 종로구 사간동',
 '서울특별시 종로구 송현동',
 '서울특별시 종로구 가회동',
 '서울특별시 종로구 재동',
 '서울특별시 종로구 계동',
 '서울특별시 종로구 원서동',
 '서울특별시 종로구 훈정동',
 '서울특별시 종로구 묘동',
 '서울특별시 종로구 봉익동',
 '서울특별시 종로구 돈의동',
 '서울특별시 종로구 장사동',
 '

In [186]:
name_list[1][-1]

'구'

In [190]:
name_list[1].split()[1]

'종로구'

In [201]:
goo = []
for name in name_list:
    if name[-1] == '구':
        name_split = name.split(' ')
        goo.append(name_split[1])

In [203]:
goo

['종로구',
 '중구',
 '용산구',
 '성동구',
 '광진구',
 '동대문구',
 '중랑구',
 '성북구',
 '강북구',
 '도봉구',
 '노원구',
 '은평구',
 '서대문구',
 '마포구',
 '양천구',
 '강서구',
 '구로구',
 '금천구',
 '영등포구',
 '동작구',
 '관악구',
 '서초구',
 '강남구',
 '송파구',
 '강동구']

In [191]:
#2번째 방법
df = lawd_cd_seoule

In [195]:
df['name'].str.split().str[1].dropna().unique()

array(['종로구', '중구', '용산구', '성동구', '광진구', '동대문구', '중랑구', '성북구', '강북구',
       '도봉구', '노원구', '은평구', '서대문구', '마포구', '양천구', '강서구', '구로구', '금천구',
       '영등포구', '동작구', '관악구', '서초구', '강남구', '송파구', '강동구'], dtype=object)

In [197]:
seoul_go = df[df['name'].str.contains('서울특별시')]['name'].str.split().str[1].dropna().unique()

array(['종로구', '중구', '용산구', '성동구', '광진구', '동대문구', '중랑구', '성북구', '강북구',
       '도봉구', '노원구', '은평구', '서대문구', '마포구', '양천구', '강서구', '구로구', '금천구',
       '영등포구', '동작구', '관악구', '서초구', '강남구', '송파구', '강동구'], dtype=object)

- **(필수) 특정월에 서울시 전체 구의 데이터를 한번에 가져오기(입력값: 연월 202402)**

In [208]:
base = "202510"
year = int(base[:4])
month = int(base[4:])
result = []

for i in range(12):
    result.append(f"{year}{month:02d}")

340282366920938463463374607431768211456

In [209]:
import requests

url = 'http://apis.data.go.kr/1360000/MidFcstInfoService/getMidFcst'
params ={'serviceKey' : '서비스키', 'pageNo' : '1', 'numOfRows' : '10', 'dataType' : 'XML', 'stnId' : '108', 'tmFc' : '201310170600' }

response = requests.get(url, params=params)
print(response.content)

b'Unauthorized\n'
